In [1]:
from corrosions import Sync
from corrosions.utils import calculate_distance
import numpy as np

In [2]:
sync = Sync(
    area="Bogor",
    segment="Jonggol-Cimanggis",
    year=2024,
    pipe_diameter=16,
    length=9.95,
    acvg_dcvg_file=r"D:\Projects\corrosions\tests\acvg-dcvg-jonggol-cimanggis-16.xlsx",
    normalized_cips_file=r"D:\Projects\corrosions\tests\normalize\cips\excel\cips-bgr-16-inch-jonggol-cimanggis-9-95-km-survey-data.xlsx",
    normalized_pcm_file=r"D:\Projects\corrosions\tests\normalize\pcm\excel\pcm-bgr-16-inch-jonggol-cimanggis-9-95-km-sequential-file.xlsx",
    verbose=True,
)

In [3]:
df_pcm = sync.df_pcm
df_acvg_dcvg = sync.df_acvg_dcvg

In [4]:
df_pcm.head()

,Start Symbol,Format,Version,PCMx Operating mode,Locator Frequency,Alpha display,Depth (m),Depth (ft),Depth to pipe center (m),Depth to pipe center (ft),...,Ext GPS no. of satellites,Ext GPS dilution,Ext GPS altitude,Pipe Diameter,Survey name (0-100),Comment (0-100),dbma,Distance,Real Distance,Direction
Index,,,,,,,,,,,,,,,,,,,,,
6904,PCMx,9,1,19,512,LFCD,1.68,5.511811,1.8848,6.183727,...,0,0,0,0.4096,BGR 16 INCH ( JONGGOL CIMANGGIS) P6,Stasiun Gas Pembagi Cimanggis 2,39.490234,10.562922,0.000000,DECREASING
6903,PCMx,9,1,19,512,LFCD,1.74,5.708662,1.9448,6.380578,...,0,0,0,0.4096,BGR 16 INCH ( JONGGOL CIMANGGIS) P6,NaN,39.627310,12.770280,10.562922,DECREASING
6902,PCMx,9,1,19,512,LFCD,1.66,5.446194,1.8648,6.118110,...,0,0,0,0.4096,BGR 16 INCH ( JONGGOL CIMANGGIS) P6,NaN,39.806777,11.897857,23.333202,DECREASING
6901,PCMx,9,1,19,512,LFCD,1.70,5.577428,1.9048,6.249344,...,0,0,0,0.4096,BGR 16 INCH ( JONGGOL CIMANGGIS) P6,NaN,39.895139,11.553697,35.231059,DECREASING
6900,PCMx,9,1,19,512,LFCD,1.66,5.446194,1.8648,6.118110,...,0,0,0,0.4096,BGR 16 INCH ( JONGGOL CIMANGGIS) P6,NaN,39.545324,12.895244,46.784756,DECREASING


In [6]:
df_acvg_dcvg

,segment_code,diameter,anomaly_location,surface_condition,drop_pcm,on_potential,off_potential,survey_dcvg,survey_acvg,latitude,longitude,ir_drop,pipe_depth,result_acvg,protection,comment,uncertain,uncertain_description
0,jonggol-cimanggis-16,16,NaN,NaN,NaN,-0.96418,NaN,45457,45445,-6.404611,106.962472,NaN,NaN,44,NaN,NaN,NaN,NaN
1,jonggol-cimanggis-16,16,sebrang THRILL SPILL CUSTOM,beton,NaN,-0.99236,NaN,45457,45445,-6.403722,106.960778,NaN,NaN,56,NaN,NaN,NaN,NaN
2,jonggol-cimanggis-16,16,depan Bank DMM,parit,NaN,-0.75544,NaN,45457,45445,-6.378194,106.920778,43.75,NaN,89,NaN,NaN,NaN,NaN
3,jonggol-cimanggis-16,16,depan Sate & Sop Kita,beton,NaN,-0.92539,NaN,45457,45445,-6.376799,106.918248,2.10,NaN,51,NaN,NaN,NaN,NaN
4,jonggol-cimanggis-16,16,depan Ramayana Prime,trotoar,NaN,-0.91251,NaN,45457,45445,-6.375500,106.915528,NaN,NaN,53,NaN,NaN,NaN,NaN
5,jonggol-cimanggis-16,16,depan tulis MR DIY,trotoar beton,NaN,-0.92921,NaN,45457,45445,-6.375389,106.915278,10.61,NaN,53,NaN,NaN,NaN,NaN
6,jonggol-cimanggis-16,16,depan jual durian/Manna & Salwa,beton,NaN,-0.93930,NaN,45457,45445,-6.375861,106.906556,10.67,NaN,65,NaN,NaN,NaN,NaN
7,jonggol-cimanggis-16,16,depan Huben Inside,beton trotoar,NaN,-1.00088,NaN,45457,45445,-6.375833,106.905361,1.29,NaN,45,NaN,NaN,NaN,NaN


In [ ]:
acvg_dcvg_coordinates = df_acvg_dcvg[["latitude", "longitude"]]

In [ ]:
pcm_coordinates = df_pcm[["Int GPS Latitude", "Int GPS Longitude", "Real Distance"]]

In [ ]:
acvg_dvcg_distance = []
closest_pcm_distance = []
closest_pcm_index = []
closest_pcm_latitude = []
closest_pcm_longitude = []
closest_pcm_real_distance = []
for index, row_acvg_dcvg in df_acvg_dcvg.iterrows():
    distances = []
    lat2 = row_acvg_dcvg["latitude"]
    lon2 = row_acvg_dcvg["longitude"]

    for _, pcm in pcm_coordinates.iterrows():
        lat1 = pcm["Int GPS Latitude"]
        lon1 = pcm["Int GPS Longitude"]
        distance = calculate_distance(lat1, lon1, lat2, lon2)
        distances.append(distance)

    np_distances = np.array(distances)
    distance_min = np.min(np_distances)

    acvg_dvcg_distance.append(pcm_coordinates.iloc[np_distances.argmin()]["Real Distance"] + distance_min)
    closest_pcm_distance.append(distance_min)
    closest_pcm_index.append(np_distances.argmin())
    closest_pcm_latitude.append(pcm_coordinates.iloc[np_distances.argmin()]["Int GPS Latitude"])
    closest_pcm_longitude.append(pcm_coordinates.iloc[np_distances.argmin()]["Int GPS Longitude"])
    closest_pcm_real_distance.append(pcm_coordinates.iloc[np_distances.argmin()]["Real Distance"])

df_acvg_dcvg["real_distance"] = acvg_dvcg_distance
df_acvg_dcvg["closest_pcm_distance"] = closest_pcm_distance
df_acvg_dcvg["closest_pcm_index"] = closest_pcm_index
df_acvg_dcvg["closest_pcm_latitude"] = closest_pcm_latitude
df_acvg_dcvg["closest_pcm_longitude"] = closest_pcm_longitude
df_acvg_dcvg["closest_pcm_real_distance"] = closest_pcm_real_distance


In [ ]:
df_acvg_dcvg